In [1]:
import os
os.environ["MLFLOW_TRACKING_URI"] = "https://dagshub.com/gauraviiita/datascienceproject.mlflow"
os.environ["MLFLOW_TRACKING_USERNAME"] = "gauraviiita"
os.environ["MLFLOW_TRACKING_PASSWORD"] = "c26f8f69cda9955500f4bb478fcb744168b2cb35"

In [2]:
import os
%pwd

'/Users/gauravkumaryadav/Desktop/gaurav/learning/MLOPs/end2endproject/datascienceproject/research'

In [3]:
os.chdir("../")
%pwd

'/Users/gauravkumaryadav/Desktop/gaurav/learning/MLOPs/end2endproject/datascienceproject'

In [4]:
from dataclasses import dataclass
from pathlib import Path


@dataclass
class ModelEvaluationConfig:
    root_dir: Path
    test_data_path: Path
    model_path: Path
    all_params: dict
    metric_file_name: Path
    target_column: str
    mlflow_uri: str
    

In [5]:
from src.datascience.constants import *
from src.datascience.utils.common import read_yaml, create_directories, save_json

In [6]:
class ConfigurationManager:
    def __init__(self,
                 config_filepath=CONFIG_FILE_PATH,
                 param_filepath=PARAM_FILE_PATH,
                 schema_filepath=SCHEMA_FILE_PATH):
        
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(param_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_model_evaluation_config(self)-> ModelEvaluationConfig:
        config = self.config.model_evaluation
        params = self.params.ElasticNet
        schema = self.schema.TARGET_COLUMN

        create_directories([config.root_dir])

        model_evaluation_config = ModelEvaluationConfig(
            root_dir = config.root_dir,
            test_data_path = config.test_data_path,
            model_path = config.model_path,
            all_params = params,
            metric_file_name = config.metric_file_name,
            target_column = schema.name,
            mlflow_uri = "https://dagshub.com/gauraviiita/datascienceproject.mlflow"
        )
        return model_evaluation_config

In [7]:
import os
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from urllib.parse import urlparse
import mlflow
import mlflow.sklearn
import numpy as np
import joblib


In [8]:
class ModelEvaluation:
    def __init__(self, config: ModelEvaluationConfig):
        self.config = config

    def eval_metrics(self, actual, pred):
        rmse = np.sqrt(mean_squared_error(actual, pred))
        mae = mean_absolute_error(actual, pred)
        r2 = r2_score(actual, pred)
        return rmse, mae, r2
    
    def log_into_mlflow(self):
        test_data = pd.read_csv(self.config.test_data_path)
        model = joblib.load(self.config.model_path)

        test_x = test_data.drop([self.config.target_column], axis=1)
        test_y = test_data[[self.config.target_column]]

        mlflow.set_registry_uri(self.config.mlflow_uri)
        tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme

        with mlflow.start_run():
            predicted_qualities = model.predict(test_x)

            (rmse, mae, r2) = self.eval_metrics(test_y, predicted_qualities)

            ## Saving metrics as local
            scores = {"rmse": rmse, "mae": mae, "r2": r2}
            save_json(path=Path(self.config.metric_file_name), data=scores)

            mlflow.log_params(self.config.all_params)

            mlflow.log_metric("rmse", rmse)
            mlflow.log_metric("r2", r2)
            mlflow.log_metric("mae", mae)

            ## Model registry does not work with file store
            if tracking_url_type_store != "file":
                mlflow.sklearn.log_model(model, "model", registered_model_name="ElasticNetModel")
            else:
                mlflow.sklearn.log_model(model, "model")

In [9]:
try:
    config = ConfigurationManager()
    model_evaluation_config = config.get_model_evaluation_config()
    model_evaluation = ModelEvaluation(config=model_evaluation_config)
    model_evaluation.log_into_mlflow()
except Exception as e:
    raise e

[2026-08-15 10:26:35,941: INFO: common: yaml file: config/config.yaml loaded successfully]
[2026-08-15 10:26:35,942: INFO: common: yaml file: param.yaml loaded successfully]
[2026-08-15 10:26:35,945: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-08-15 10:26:35,946: INFO: common: created directory at: artifacts]
[2026-08-15 10:26:35,947: INFO: common: created directory at: artifacts/model_evaluation]
[2026-08-15 10:26:37,240: INFO: common: json file saved at: artifacts/model_evaluation/metrics.json]


/Users/gauravkumaryadav/Desktop/gaurav/learning/MLOPs/end2endproject/datascienceproject/venv/lib/python3.10/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/gauravkumaryadav/Desktop/gaurav/learning/MLOPs/end2endproject/datascienceproject/venv/lib/python3.10/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/gauravkumaryadav/Desktop/gaurav/learning/MLOPs/end2endproject/datascienceproject/venv/lib/python3.10/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_ + self.intercept_
2026/08/15 10:26:38 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
Successfully registered model 'ElasticNetModel'.
2026/08/15 10:26:47 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model versi

🏃 View run righteous-mare-814 at: https://dagshub.com/gauraviiita/datascienceproject.mlflow/#/experiments/0/runs/14c260217ca147e58826f822cef1ea05
🧪 View experiment at: https://dagshub.com/gauraviiita/datascienceproject.mlflow/#/experiments/0
